# Mosaico ingestion demo: medkit fault snapshots as queryable forensic data

**What this notebook shows.** Robot stack runs in Docker. A fault is injected on the LiDAR (high noise). The medkit gateway detects it via the standard `/diagnostics` ROS topic, confirms the fault, and flushes its 15-second pre-fault + 10-second post-fault ring buffer to an `.mcap` file. A small Python bridge (separate process, talking Apache Arrow Flight) picks up the SSE event, downloads the bag from the gateway REST API, and ingests it into mosaicod.

By the time you run this notebook, the bag is already a Mosaico **Sequence** with **typed**, **queryable** topics: `LaserScan`, `Imu`, and `GPS` ontologies (`/diagnostics` still rides along in the MCAP but has no Mosaico adapter yet, so it lands silently). The raw camera topic `/sensors/image_raw` is intentionally left out of the snapshot - at 30 Hz uncompressed it would dominate the bag - so the catalog entry is a few MB rather than a few hundred. Mosaico's Image adapter does ship, so dropping in a `CompressedImage` topic later is a config change, not a code change.

We never recorded the robot 24/7 - we only kept the 25 seconds around the fault, but those 25 seconds are now indexed, schema-validated, cross-referenceable, and ready for `.Q` queries.

Pipeline:

```
lidar_sim --(LaserScan)--> /sensors/scan --[ring buffer 15s pre + 10s post]
                                                      |
  noise injection                                     v
       |             confirm fault              flush to .mcap
       v                  ^                          |
  diagnostic (ERROR) --> diagnostic_bridge --> fault_manager --> SSE /faults/stream
                                                                       |
                                                                       v
                                          bridge downloads .mcap via REST
                                                                       |
                                                                       v
                                            RosbagInjector --> mosaicod (Arrow Flight)
```

**License-safe pattern.** mosaicod is the unmodified upstream Docker image. The bridge is a separate Python process talking the public Apache Arrow Flight protocol via Mosaico's own SDK. We never link or modify mosaicod or its Rust crates.

## Setup

Run this notebook from the host with the Mosaico integration stack running. Ports the stack publishes:

- `localhost:18080` - medkit gateway (REST + SSE)
- `localhost:16726` - mosaicod (Apache Arrow Flight)

Install the SDK plus the two plotting deps used below:

```bash
pip install 'mosaicolabs>=0.4.0,<0.5' matplotlib pandas
```

If you skipped installing locally and only have the docker stack, you can still execute the cells inside the bridge container with `docker compose exec bridge python -c '...'`.

In [ ]:
# IMPORTANT: import the futures.laser module so the LaserScan ontology
# is registered in Mosaico's global ontology registry. Without this the
# topic reader raises 'No ontology registered with tag laser_scan'.
import mosaicolabs.models.futures.laser  # noqa: F401

from mosaicolabs import IMU, MosaicoClient, QueryOntologyCatalog, QueryTopic
from mosaicolabs.models.futures import LaserScan

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MOSAICO_HOST = "localhost"
MOSAICO_PORT = 16726

client = MosaicoClient.connect(host=MOSAICO_HOST, port=MOSAICO_PORT)
print("Connected to mosaicod")

## 1. List ingested sequences

Each fault snapshot landed as a `Sequence` named `{source_demo}_{robot_id}_{fault_code}_{timestamp}_{event_id}` (for example `fleet_demo_robot-01-warehouse-A_LIDAR_SIM_1776369146_1`). The `robot_id` component keeps fleet runs from colliding when two robots happen to hit `event_id=1` at the same wall-clock second. Sequence metadata includes the fault code, severity, timestamp, and reporting source - all queryable later from any `.Q` filter.

In [ ]:
sequence_names = sorted(client.list_sequences())
print(f"Found {len(sequence_names)} sequences\n")

rows = []
for name in sequence_names:
    sh = client.sequence_handler(name)
    if sh is None:
        continue
    rows.append(
        {
            "sequence": name,
            "size_MB": round(sh.total_size_bytes / (1024 * 1024), 2),
            "topics": ", ".join(sorted(sh.topics)),
        }
    )

pd.DataFrame(rows)

## 2. Query 1: which sequences carry LaserScan data?

We use Mosaico's type-safe ontology query to find every topic that carries `LaserScan` payloads, anywhere in the catalog. This is the kind of cross-sequence search that a folder of `.mcap` files can never give you - you would otherwise grep through filenames and pray.

In [ ]:
results = client.query(
    QueryTopic().with_ontology_tag(LaserScan.ontology_tag()),
)
for item in results:
    print(item.sequence.name)
    for topic in item.topics:
        print(f"   {topic.name}")

## 3. Query 2: pull /sensors/scan from a chosen sequence

Pick the most recent LIDAR_SIM fault sequence and walk through its scans. Each item arrives as a typed `LaserScan` with all the standard fields: `angle_min`, `angle_max`, `range_min`, `range_max`, `ranges`, `intensities`, plus `frame_id` from the original ROS message header.

In [ ]:
lidar_sequences = [n for n in sequence_names if "LIDAR_SIM" in n]
if not lidar_sequences:
    raise SystemExit(
        "No LIDAR_SIM sequences yet. Run ./scripts/trigger-fault.sh first."
    )
target_sequence = lidar_sequences[-1]
print(f"Target sequence: {target_sequence}")

sh = client.sequence_handler(target_sequence)
scan_topic = sh.get_topic_handler("/sensors/scan")

scan_records = []
for item in scan_topic.get_data_streamer():
    ls = item.data
    arr = np.array(ls.ranges, dtype=float)
    finite = arr[np.isfinite(arr)]
    scan_records.append(
        {
            "timestamp_ns": item.timestamp_ns,
            "frame_id": item.frame_id,
            "range_min": float(finite.min()) if finite.size else math.nan,
            "range_max": float(finite.max()) if finite.size else math.nan,
            "range_mean": float(finite.mean()) if finite.size else math.nan,
            "range_std": float(finite.std()) if finite.size else math.nan,
            "n_beams": len(ls.ranges),
        }
    )

scan_df = pd.DataFrame(scan_records)
scan_df["t_s"] = (scan_df["timestamp_ns"] - scan_df["timestamp_ns"].iloc[0]) / 1e9
scan_df.head()

## 4. Query 3: pull /sensors/imu from the same sequence

Same pattern, different ontology. We pull the IMU acceleration and angular velocity to check whether the robot was moving when the LiDAR fault appeared. If the IMU shows zero horizontal acceleration and zero rotation throughout the 25 s window, the fault is in the sensor, not caused by robot motion.

In [ ]:
imu_topic = sh.get_topic_handler("/sensors/imu")
imu_records = []
for item in imu_topic.get_data_streamer():
    imu = item.data
    imu_records.append(
        {
            "timestamp_ns": item.timestamp_ns,
            "acc_x": imu.acceleration.x,
            "acc_y": imu.acceleration.y,
            "acc_z": imu.acceleration.z,
        }
    )
imu_df = pd.DataFrame(imu_records)
imu_df["t_s"] = (imu_df["timestamp_ns"] - scan_df["timestamp_ns"].iloc[0]) / 1e9
imu_df.head()

## 5. The money shot plot

Three subplots on the same time axis:

1. **LaserScan range_std** - jumps when noise injection hits. This is the fault signature.
2. **LaserScan range_mean** - shifts when drift injection hits, stays stable under noise. Different failure modes produce different shapes.
3. **IMU acceleration_z** - sits at ~9.8 m/s^2 the whole time. A quick sanity check that the platform was level, not tilted or falling. This is a first-pass visual - a proper stationarity check needs all six IMU axes (see section 6 below for the full compound `.Q` query).

This cross-topic correlation is what a structured, schema-indexed catalog enables.

In [ ]:
from pathlib import Path

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(11, 7))

axes[0].plot(scan_df["t_s"], scan_df["range_std"], color="crimson", linewidth=1.6)
axes[0].set_ylabel("LaserScan\nrange std [m]")
axes[0].set_title(f"{target_sequence}")
axes[0].grid(alpha=0.3)

axes[1].plot(scan_df["t_s"], scan_df["range_mean"], color="steelblue", linewidth=1.6)
axes[1].set_ylabel("LaserScan\nrange mean [m]")
axes[1].grid(alpha=0.3)

axes[2].plot(imu_df["t_s"], imu_df["acc_z"], color="forestgreen", linewidth=1.0)
axes[2].axhline(9.81, color="black", linestyle=":", alpha=0.7, label="1 g")
axes[2].set_ylabel("IMU acc_z\n[m/s^2]")
axes[2].set_xlabel("time within snapshot [s]")
axes[2].grid(alpha=0.3)
axes[2].legend(loc="upper right")

plt.tight_layout()
_out_dir = Path("../docs")
_out_dir.mkdir(parents=True, exist_ok=True)
_out_path = _out_dir / "lidar_noise_plot.png"
plt.savefig(_out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved plot to {_out_path}")

## 6. Compound .Q filter: find snapshots where the robot was stationary

A single `acc_z ~ 9.81` check only confirms the sensor is level, not that the robot is parked. A proper stationarity check needs all six IMU degrees of freedom: linear acceleration near zero on X/Y (no horizontal motion), near 1 g on Z (level, not falling), and angular velocity near zero on all axes (not rotating).

Mosaico's `QueryOntologyCatalog` supports compound expressions with implicit AND, so we can push all six conditions into one server-side query.

In the fleet variant this query is not cosmetic: the fleet injector rotates robot-02 (IMU `drift_rate=0.3 rad/s`) while triggering the same LIDAR noise as robot-01, so robot-02's `angular_velocity.z` sits around 0.3 rad/s. The `between(-0.1, 0.1)` predicate excludes that sequence, leaving only the two stationary LIDAR faults for downstream content comparison (§7). Step 2 doing real work, not just showing the SDK surface.

In [ ]:
stationary = client.query(
    QueryOntologyCatalog(
        IMU.Q.acceleration.x.between(-0.5, 0.5),      # no horizontal accel
        IMU.Q.acceleration.y.between(-0.5, 0.5),      # no horizontal accel
        IMU.Q.acceleration.z.between(9.5, 10.1),      # gravity only, level
        IMU.Q.angular_velocity.x.between(-0.1, 0.1),  # no roll
        IMU.Q.angular_velocity.y.between(-0.1, 0.1),  # no pitch
        IMU.Q.angular_velocity.z.between(-0.1, 0.1),  # no yaw
        include_timestamp_range=True,
    ),
)
print(f"Sequences matching full stationarity check:\n")
for item in stationary:
    print(item.sequence.name)
    for topic in item.topics:
        if topic.timestamp_range:
            print(
                f"   {topic.name}  {topic.timestamp_range.start}..{topic.timestamp_range.end}"
            )

## 7. Fleet comparison: noise vs drift

The fleet variant triggers LIDAR faults on all three robots (both noise and drift signatures under the same `LIDAR_SIM` fault code). The previous cell's compound IMU query already excluded robot-02 (which was rotating during its fault window). The remaining two stationary LIDAR sequences, robot-01 (noise injection) and robot-03 (drift injection), have the same metadata tag but completely different scan-level signatures. Pulling the actual scan statistics from Mosaico reveals those two distinct failure modes.

Skip this section if you only ran the single-robot variant (it needs 2+ LIDAR_SIM sequences).

In [ ]:
from pathlib import Path

# Use the stationary sequences from §6 - the IMU query already excluded robot-02
# (which was rotating during its fault). The two remaining stationary LIDAR
# sequences are robot-01 (noise injection) and robot-03 (drift injection).
stationary_names = {item.sequence.name for item in stationary}
stationary_lidar = [name for name in lidar_sequences if name in stationary_names]

if len(stationary_lidar) >= 2:
    def _pull_scan_stats(seq_name):
        _sh = client.sequence_handler(seq_name)
        _topic = _sh.get_topic_handler("/sensors/scan")
        recs = []
        for item in _topic.get_data_streamer():
            arr = np.array(item.data.ranges, dtype=float)
            fin = arr[np.isfinite(arr)]
            recs.append({
                "timestamp_ns": item.timestamp_ns,
                "range_mean": float(fin.mean()) if fin.size else math.nan,
                "range_std": float(fin.std()) if fin.size else math.nan,
            })
        df = pd.DataFrame(recs)
        df["t_s"] = (df["timestamp_ns"] - df["timestamp_ns"].iloc[0]) / 1e9
        return df

    # Sort so robot-01 is consistently on the left, robot-03 on the right.
    stationary_lidar = sorted(stationary_lidar)
    seq_a, seq_b = stationary_lidar[0], stationary_lidar[1]
    df_a = _pull_scan_stats(seq_a)
    df_b = _pull_scan_stats(seq_b)

    fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex="col")

    axes[0, 0].plot(df_a["t_s"], df_a["range_std"], color="crimson", lw=1.4)
    axes[0, 0].set_ylabel("range std [m]")
    axes[0, 0].set_title(seq_a, fontsize=9)
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(df_b["t_s"], df_b["range_std"], color="crimson", lw=1.4)
    axes[0, 1].set_title(seq_b, fontsize=9)
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(df_a["t_s"], df_a["range_mean"], color="steelblue", lw=1.4)
    axes[1, 0].set_ylabel("range mean [m]")
    axes[1, 0].set_xlabel("time [s]")
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(df_b["t_s"], df_b["range_mean"], color="steelblue", lw=1.4)
    axes[1, 1].set_xlabel("time [s]")
    axes[1, 1].grid(alpha=0.3)

    fig.suptitle(
        "Same fault_code (LIDAR_SIM), different root cause",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    _out_dir = Path("../docs")
    _out_dir.mkdir(parents=True, exist_ok=True)
    _out_path = _out_dir / "fleet_comparison.png"
    plt.savefig(_out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Left: {seq_a}\nRight: {seq_b}\nSaved to {_out_path}")
else:
    print(
        f"Only {len(stationary_lidar)} stationary LIDAR_SIM sequence(s) "
        f"(of {len(lidar_sequences)} total). Run the fleet variant and inject "
        f"faults on robot-01 and robot-03 for the comparison."
    )

In [ ]:
client.close()